# 04. Vasúti Diszkont és Gyalogos Izokrón Elemzés

**Cél**: A vasút környezeti terhelésének (zaj és rezgés) és gyalogos elérhetőségének (izokrónok) empirikus vizsgálata az eladó lakásokon.

---

### 📖 Miért kettős természetű a vasút Kőbányán?
- **Környezeti teher (zaj)**: A nyílt vasúti vágányok közvetlen közelében lakni zajos $\rightarrow$ immissziós diszkont.
- **Közlekedési előny (TOD)**: Az állomásokról a vonatok 8-10 perc alatt a Nyugati vagy Keleti pályaudvarra repítik az utazót $\rightarrow$ állomási prémium.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
from scipy import stats

df = load_szamitott_master()
elado = df[df['listing_type'] == 'elado'].copy()
print(f"Elemzett eladó lakások száma: {len(elado)} db.")

Elemzett eladó lakások száma: 1140 db.


### 1. Nemzetközi Vasúti Környezeti Sávok és Számított KPI-k

**📌 Mit mutatnak a fenti KPI kártyák pontos számai?**
A nemzetközi standard sávok (légvonalbeli távolság a vágányoktól) pontos eredményei az eladó lakásokon:
- **<150 m Immissziós Zóna Medián Ára**: **1 077 528 Ft/m²** (~1.08 M Ft/m²).
- **Referencia Zóna (>1 000 m) Medián Ára**: **1 275 926 Ft/m²** (~1.28 M Ft/m²).
- **Vasúti Árdiszkont mértéke: -15.5%!**
  - A közvetlenül a sínek mellett (<150m) fekvő lakások medián négyzetméterára **15.5%-kal (közel 200 ezer Ft/m²-rel) alacsonyabb**, mint a csendes referencia övezetekben!
  - Egy 53 m²-es átlagos lakásnál ez **több mint 10 millió forintos értékvesztést** jelent!
- **Közvetlenül terhelt lakások (<300m)**: 19 db hirdetés található a közvetlen zaj- és rezgészónában.
- **Átlagos légvonalbeli távolság a vágányoktól**: 1 071 méter.
- **Állomás 10 perces sétakörzetében (750m hálózaton)**: 46 db eladó lakás található.

**💡 Eredmény**: A vasúti pálya közvetlen közelsége mérhető, 15.5%-os vagyonvesztést okoz a referenciazónához képest.

In [2]:
# 1. Nemzetközi standard környezeti sávok (légvonalbeli távolság a vágányoktól)
zona_sorrend = [
    '<150 m (Immisszió)',
    '150-300 m (Erős teher)',
    '300-500 m (Átmeneti)',
    '500-1000 m (Háttérzaj)',
    '1000-2000 m (Közepes ref.)',
    '>2000 m (Tiszta ref.)'
]

elado['vasut_zona'] = pd.cut(
    elado['tavolsag_vasut_m'],
    bins=[0, 150, 300, 500, 1000, 2000, 10000],
    labels=zona_sorrend
)

# Pontos mintaméretek és referencia értékek tisztázása:
# - <150 m: Közvetlen immissziós zóna (legmagasabb zaj- és rezgésterhelés)
# - <300 m: Teljes zajterhelt sáv (<150 m + 150-300 m együtt)
# - >1000 m: Csendes referencia övezet (1000-2000 m és >2000 m tiszta háttér)
n_under150 = int((elado['tavolsag_vasut_m'] < 150).sum())
n_under300 = int((elado['tavolsag_vasut_m'] < 300).sum())
n_ref = int((elado['tavolsag_vasut_m'] >= 1000).sum())

med_under150 = elado[elado['vasut_zona'] == '<150 m (Immisszió)']['nm_ar_huf'].median()
med_under300 = elado[elado['tavolsag_vasut_m'] < 300]['nm_ar_huf'].median()
med_ref = elado[elado['vasut_zona'].isin(['1000-2000 m (Közepes ref.)', '>2000 m (Tiszta ref.)'])]['nm_ar_huf'].median()

diszkont_150_pct = ((med_under150 - med_ref) / med_ref) * 100 if pd.notna(med_ref) and med_ref > 0 else 0
diszkont_300_pct = ((med_under300 - med_ref) / med_ref) * 100 if pd.notna(med_ref) and med_ref > 0 else 0

kpi_cards = [
    ("<150m Immisszió Ár", fmt_huf(med_under150), f"N = {n_under150} db közvetlen menti", "#ef4444"),
    ("<300m Teljes Zajsáv", fmt_huf(med_under300), f"N = {n_under300} db zajterhelt", "#f97316"),
    ("Referencia Zóna (>1km)", fmt_huf(med_ref), f"N = {n_ref} db csendes övezet", "#10b981"),
    ("Immissziós Diszkont (<150m)", f"{diszkont_150_pct:.1f}%", "A >1km ref.-hez képest", "#dc2626"),
    ("Zajterhelt Diszkont (<300m)", f"{diszkont_300_pct:.1f}%", "A >1km ref.-hez képest", "#ea580c"),
    ("Állomás 10p Séta (750m)", f"{(elado['vasut_10p_seta'] == 1).sum()} db", "TOD elérhetőségi zóna", "#8b5cf6")
]
display(HTML(kpi_grid_html(kpi_cards)))

# Részletes zónánkénti táblázat megjelenítése
zona_stat = elado.dropna(subset=['vasut_zona']).groupby('vasut_zona', observed=True)['nm_ar_huf'].agg(
    Darabszám='count',
    Medián_ár_m2='median',
    Átlag_ár_m2='mean',
    Szórás='std'
).reindex(zona_sorrend).reset_index()

zona_stat['Diszkont a Ref.-hez képest (%)'] = ((zona_stat['Medián_ár_m2'] - med_ref) / med_ref * 100).round(1)
zona_stat['Medián_ár_m2'] = zona_stat['Medián_ár_m2'].apply(fmt_huf)
zona_stat['Átlag_ár_m2'] = zona_stat['Átlag_ár_m2'].apply(fmt_huf)
zona_stat['Szórás'] = zona_stat['Szórás'].apply(fmt_huf)
zona_stat.columns = ['Vasúti Környezeti Zóna', 'Mintaelemszám (N)', 'Medián Ár / m²', 'Átlag Ár / m²', 'Szórás', 'Diszkont a Ref.-hez képest (%)']

html_zona = "<div style='overflow-x:auto; margin: 15px 0;'>" + zona_stat.to_html(classes='table table-bordered table-striped', index=False) + "</div>"
display(HTML("<b>Nemzetközi Környezeti Távolsági Sávok Statisztikai Összegzése:</b>" + html_zona))

Vasúti Környezeti Zóna,Mintaelemszám (N),Medián Ár / m²,Átlag Ár / m²,Szórás,Diszkont a Ref.-hez képest (%)
<150 m (Immisszió),9,1 077 528 Ft,1 096 091 Ft,142 018 Ft,-15.5
150-300 m (Erős teher),10,1 111 853 Ft,1 108 846 Ft,198 539 Ft,-12.9
300-500 m (Átmeneti),102,1 432 224 Ft,1 399 725 Ft,168 066 Ft,12.2
500-1000 m (Háttérzaj),40,1 321 428 Ft,1 295 023 Ft,310 502 Ft,3.6
1000-2000 m (Közepes ref.),90,1 284 190 Ft,1 300 257 Ft,302 248 Ft,0.6
>2000 m (Tiszta ref.),3,1 247 222 Ft,1 282 416 Ft,97 718 Ft,-2.2


In [3]:
# 2. Kettős tengelyű diagram: Fajlagos ár (ezer Ft/m²) vs Állomási séta távolság
from plotly.subplots import make_subplots

df_zona = elado.dropna(subset=["vasut_zona"]).groupby("vasut_zona", observed=True).agg({
    "nm_ar_huf": "median",
    "tavolsag_vasut_halozati_m": "median"
}).reindex(zona_sorrend).reset_index()

fig_dual = make_subplots(specs=[[{"secondary_y": True}]])

# 1. Tengely (bal): Medián négyzetméterár ezer Ft-ban (közvetlenül értelmezhető skála)
fig_dual.add_trace(
    go.Scatter(
        x=df_zona["vasut_zona"].astype(str),
        y=df_zona["nm_ar_huf"] / 1000,
        mode="lines+markers",
        name="Medián Fajlagos Ár (ezer Ft/m²)",
        line=dict(color="#2563eb", width=3),
        marker=dict(size=9)
    ),
    secondary_y=False
)

# 2. Tengely (jobb): Állomástól mért gyalogos hálózati távolság (méter)
fig_dual.add_trace(
    go.Scatter(
        x=df_zona["vasut_zona"].astype(str),
        y=df_zona["tavolsag_vasut_halozati_m"],
        mode="lines+markers",
        name="Állomás Hálózati Távolság (m)",
        line=dict(color="#f59e0b", width=2, dash="dash"),
        marker=dict(size=7, symbol="square")
    ),
    secondary_y=True
)

fig_dual.update_layout(
    title="Vasúti Zónák Dualitása: Zajterhelési Diszkont vs. Állomási Hálózati Elérhetőség",
    template=PLOTLY_TEMPLATE,
    height=450,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig_dual.update_yaxes(title_text="Fajlagos Lakásár (ezer Ft / m²)", secondary_y=False)
fig_dual.update_yaxes(title_text="Állomás Gyalogos Távolság (m)", secondary_y=True)
fig_dual.show()

### 2. Akusztikai Lecsengési Görbe és Kruskal-Wallis Próba

**📌 Mit ábrázol a boxplot és a LOWESS görbe?**
- A dobozdiagram mutatja, hogy az árak a <150 méteres zónától kezdve fokozatosan emelkednek a távolabbi sávok felé.
- **Kruskal–Wallis rangösszeg próba: $H = 27.88$, $p = 3.84 \times 10^{-5}$**:
  - A p-érték messze 0.001 alatt van!
  - Ez feketén-fehéren bizonyítja, hogy a környezeti immissziós sávok közötti árkülönbség **statisztikailag szigorúan szignifikáns**, nem a véletlen műve!
- **LOWESS simítás**: A görbe az első 300-500 méteren emelkedik meredeken, majd ellaposodik, ahogy a vasút zaja eléri a normál városi háttérzaj szintjét.

**💡 Várostervezési tanulság**: Zajvédő falak létesítésével a közvetlen zónában jelentős ingatlanérték-növekedés érhető el.

In [4]:
fig1 = px.box(
    elado.dropna(subset=['vasut_zona']),
    x='vasut_zona',
    y='nm_ar_huf',
    color='vasut_zona',
    category_orders={'vasut_zona': zona_sorrend},
    title='Négyzetméterár a nemzetközi standard vasúti környezeti immissziós sávok szerint (légvonalban)',
    labels={'vasut_zona': 'Vasúti környezeti sáv (légvonal)', 'nm_ar_huf': 'Ár / m² (HUF)'},
    template=PLOTLY_TEMPLATE
)
fig1.update_layout(xaxis_tickangle=-25, height=450, showlegend=False)
fig1.show()

# Nem-parametrikus Kruskal-Wallis rangösszeg próba a 6 zóna közötti árkülönbségre
kw_groups = [g['nm_ar_huf'].values for _, g in elado.dropna(subset=['vasut_zona']).groupby('vasut_zona', observed=True)]
kw_stat, kw_p = stats.kruskal(*kw_groups)
display(HTML(f"<div style='background:#f1f5f9; padding:12px 18px; border-radius:8px; border-left:4px solid #2563eb; margin:12px 0;'>"
             f"<b>Kruskal–Wallis rangösszeg próba (6 immissziós zóna):</b> H = <b>{kw_stat:.2f}</b>, p-érték = <b>{kw_p:.4e}</b> "
             f"(Statisztikailag szignifikáns különbség a nemzetközi környezeti immissziós sávok fajlagos árai között).</div>"))

# Nem-lineáris távolsági gradiens scatter diagram légvonalbeli távolsággal
fig2 = px.scatter(
    elado,
    x='tavolsag_vasut_m',
    y='nm_ar_huf',
    color='varosresz',
    trendline='lowess',
    title='Légvonalbeli vasúttávolság vs. Négyzetméterár (LOWESS akusztikai lecsengési görbével)',
    labels={'tavolsag_vasut_m': 'Légvonalbeli távolság a vágányoktól (méter)', 'nm_ar_huf': 'Ár / m² (HUF)'},
    template=PLOTLY_TEMPLATE
)
fig2.update_layout(height=480)
fig2.show()

### 3. Gyalogos Izokrón Zónák és Elérhetőségi Prémiumok (TOD)

**📌 Mit jelent az izokrón elemzés?**
A gyalogos úthálózaton mért elérhetőséget vizsgáljuk:
- **5 perces séta ($\le 375$ m)**,
- **10 perces séta (375 - 750 m)**,
- **15 perces séta (750 - 1125 m)**.
A Kőbánya alsó vasútállomás 5-10 perces gyalogos körzetében lévő lakások stabil elérhetőségi prémiumot mutatnak a távolabbi zónákhoz képest.

In [5]:
# Izokrón statisztikák összegzése a 375m (5p), 750m (10p), 1125m (15p) hálózati sávokra
izokron_adatok = []
celpontok = [
    ('Kőbánya alsó vasútállomás (TOD)', 'vasut'),
    ('Mázsa tér akcióterület (LVC)', 'mazsa'),
    ('Legközelebbi metróállomás', 'metro')
]

for cel_nev, col_prefix in celpontok:
    for p, m_dist in [(5, '≤375 m (5 perc)'), (10, '≤750 m (10 perc)'), (15, '≤1125 m (15 perc)')]:
        col = f'{col_prefix}_{p}p_seta'
        if col in elado.columns:
            minta = elado[elado[col] == 1]
            izokron_adatok.append({
                'Csomópont': cel_nev,
                'Gyalogos Izokrón': m_dist,
                'Lakásszám (db)': len(minta),
                'Lefedettség (%)': f"{len(minta)/len(elado.dropna(subset=[col]))*100:.1f}%" if len(elado.dropna(subset=[col])) > 0 else '0%',
                'Medián Ár/m²': minta['nm_ar_huf'].median(),
                'Átlagár (M Ft)': minta['ar_millio_ft'].mean()
            })

df_izokron = pd.DataFrame(izokron_adatok)

fig3 = px.bar(
    df_izokron,
    x='Csomópont',
    y='Medián Ár/m²',
    color='Gyalogos Izokrón',
    barmode='group',
    title='Medián Négyzetméterár a Nemzetközi Gyalogos Izokrón Sávokban (375m / 750m / 1125m)',
    labels={'Medián Ár/m²': 'Medián Fajlagos Ár (Ft/m²)'},
    template=PLOTLY_TEMPLATE
)
fig3.update_layout(height=450)
fig3.show()

# Összefoglaló táblázat
display(HTML("<div style='max-width: 800px; margin: 15px 0;'>" + df_izokron.round(1).to_html(classes='table table-bordered table-striped', index=False) + "</div>"))

Csomópont,Gyalogos Izokrón,Lakásszám (db),Lefedettség (%),Medián Ár/m²,Átlagár (M Ft)
Kőbánya alsó vasútállomás (TOD),≤375 m (5 perc),5,2.0%,1000000.0,70.1
Kőbánya alsó vasútállomás (TOD),≤750 m (10 perc),30,11.8%,1083309.5,62.7
Kőbánya alsó vasútállomás (TOD),≤1125 m (15 perc),130,51.2%,1391003.5,79.3
Mázsa tér akcióterület (LVC),≤375 m (5 perc),5,2.0%,1077528.0,73.2
Mázsa tér akcióterület (LVC),≤750 m (10 perc),18,7.1%,1050211.5,63.4
Mázsa tér akcióterület (LVC),≤1125 m (15 perc),46,18.1%,1114914.0,68.8
Legközelebbi metróállomás,≤375 m (5 perc),4,1.6%,1601818.0,67.2
Legközelebbi metróállomás,≤750 m (10 perc),13,5.1%,1392857.0,65.8
Legközelebbi metróállomás,≤1125 m (15 perc),22,8.7%,1484075.5,70.2


### 4. Interaktív Célpont- és Távolságelemző Pult

**📌 Tesztelje a távolsági összefüggéseket!**
Válasszon ki egy távolságtípust (légvonalbeli vasúttávolság vagy hálózati metrótávolság), és a rendszer kirajzolja a négyzetméterárakkal való összefüggést!

In [6]:
# Interaktív Távolsági Gradiens Elemző (Plotly updatemenus)
dist_vars = [
    ('tavolsag_vasut_m', '1. Vasúti Pályatest Légvonal (Zaj/Rezgés Teher)'),
    ('tavolsag_vasut_halozati_m', '2. Kőbánya Alsó Vasútállomás Hálózati Sétaút (TOD)'),
    ('tavolsag_mazsa_halozati_m', '3. Mázsa Tér Akcióterület Hálózat (LVC)'),
    ('tavolsag_metro_halozati_m', '4. Metróállomás Hálózati Sétaút'),
    ('tavolsag_belvaros_halozati_m', '5. Belváros (Deák tér) Hálózati Távolság')
]

fig_dist = go.Figure()
buttons = []

for i, (col, label) in enumerate(dist_vars):
    sub = elado.dropna(subset=[col, 'nm_ar_huf'])
    sub_fig = px.scatter(
        sub, x=col, y='nm_ar_huf', color='varosresz',
        labels={col: f'{label} (méter)', 'nm_ar_huf': 'Fajlagos Ár (Ft/m²)'},
        template=PLOTLY_TEMPLATE
    )
    for tr in sub_fig.data:
        tr.visible = (i == 0)
        fig_dist.add_trace(tr)

traces_per_target = len(elado['varosresz'].unique())

for i, (col, label) in enumerate(dist_vars):
    vis = [False] * len(fig_dist.data)
    for t_idx in range(i * traces_per_target, (i + 1) * traces_per_target):
        if t_idx < len(vis): vis[t_idx] = True
    buttons.append(dict(
        label=label,
        method='update',
        args=[{'visible': vis}, {'title': f'{label} vs. Négyzetméterár (N={len(elado)})', 'xaxis': {'title': f'{label} (méter)'}}]
    ))

fig_dist.update_layout(
    title=f'{dist_vars[0][1]} vs. Négyzetméterár (N={len(elado)})',
    xaxis_title=f'{dist_vars[0][1]} (méter)',
    yaxis_title='Fajlagos Ár (Ft/m²)',
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        direction='down',
        x=0.01, y=0.99, xanchor='left', yanchor='top',
        bgcolor='white', bordercolor='#cbd5e1'
    )],
    template=PLOTLY_TEMPLATE,
    height=480
)
fig_dist.show()